# Does seed expansion find the members it was not given?


**The test (instruction 64.12).** Half of a niche's hyperLOPIT markers, whole orthogroups at a time, are given as seeds; the other half are hidden. Seed expansion (random walk with restart over the measured networks and the measurement space) ranks the genome with the hyperLOPIT compartment excluded, and the hidden members are counted at fixed depths. The candidate universe is every gene except the seeds. **What this is not:** the genes that are not markers are not verified negatives, so precision here is a lower bound on how often a ranked gene is truly in the niche.

Built 2026-11-07.

In [1]:
import json, os, subprocess, sys
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
NICHES = ['rhoptries', 'micronemes', 'dense granules', 'IMC', 'apicoplast', 'mitochondrion']
SCRIPT = os.path.join(REPO, 'scripts', 'build_seed_expansion_benchmark_notebook.py')
def run(niche):
    out = os.path.join(OUT, 'row_' + niche.replace(' ', '_') + '.json')
    subprocess.run([sys.executable, SCRIPT, '--niche', niche, '--out', out], check=True,
                   env={**os.environ, 'OMP_NUM_THREADS': '2'})
    return json.load(open(out))
with ThreadPoolExecutor(max_workers=len(NICHES)) as pool:
    rows = list(pool.map(run, NICHES))
table = pd.DataFrame(rows)
table.to_csv(os.path.join(OUT, 'recovery.tsv'), sep='\t', index=False)
table

,niche,seeds,hidden,prevalence,hidden_in_top50,recall_at_50,lift_at_50,hidden_in_top100,recall_at_100,lift_at_100,hidden_in_top200,recall_at_200,lift_at_200
0,rhoptries,21,26,0.0032,14,0.538,87.4,21,0.808,65.6,21,0.808,32.8
1,micronemes,15,14,0.00172,11,0.786,127.7,12,0.857,69.6,12,0.857,34.8
2,dense granules,19,21,0.00259,9,0.429,69.6,12,0.571,46.4,13,0.619,25.1
3,IMC,19,19,0.00234,9,0.474,76.9,10,0.526,42.7,13,0.684,27.8
4,apicoplast,24,24,0.00296,4,0.167,27.1,7,0.292,23.7,8,0.333,13.5
5,mitochondrion,33,35,0.00432,7,0.2,32.4,12,0.343,27.8,13,0.371,15.1
